In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # 01 - Extracao incremental -> Bronze
# MAGIC
# MAGIC Busca issues e PRs do GitHub **a partir do watermark de cada repositorio** e grava o JSON cru no Bronze (append-only).
# MAGIC
# MAGIC Decisoes de desenho:
# MAGIC - A API e consultada com `since=<watermark>`, `sort=updated`, `direction=asc`.
# MAGIC - O watermark novo e o maior `updated_at` **recebido** (nunca o relogio local) e so avanca **depois** do commit no Bronze.
# MAGIC - Ordem ascendente + limite de paginas = extracao retomavel: se truncar, a proxima execucao continua de onde parou.
# MAGIC - `since` e inclusivo, entao o ultimo item volta a ser buscado. Isso gera duplicata no Bronze, resolvida pelo MERGE no Silver.

# COMMAND ----------

# MAGIC %run ./00_common

# COMMAND ----------

import requests

dbutils.widgets.text("repos", "delta-io/delta,dbt-labs/dbt-core,great-expectations/great_expectations")
dbutils.widgets.text("backfill_days", "7")
dbutils.widgets.text("max_pages_per_repo", "10")
dbutils.widgets.dropdown("full_refresh", "false", ["false", "true"])

REPOS = [r.strip() for r in dbutils.widgets.get("repos").split(",") if r.strip()]
BACKFILL_DAYS = int(dbutils.widgets.get("backfill_days"))
MAX_PAGES = int(dbutils.widgets.get("max_pages_per_repo"))
FULL_REFRESH = dbutils.widgets.get("full_refresh") == "true"

# Token e opcional. Sem token: 60 req/h. Com token: 5000 req/h.
# Para usar: crie um secret scope "github" com a chave "token".
try:
    GITHUB_TOKEN = dbutils.secrets.get("github", "token")
except Exception:
    GITHUB_TOKEN = None

print(f"Repos: {REPOS}")
print(f"Backfill (primeira carga): {BACKFILL_DAYS} dias | max paginas/repo: {MAX_PAGES} | full_refresh: {FULL_REFRESH}")
print(f"Autenticado: {GITHUB_TOKEN is not None}")

In [0]:

API = "https://api.github.com"


class GitHubRateLimit(Exception):
    pass


def _get(url, headers, params=None, retries=3):
    for attempt in range(1, retries + 1):
        try:
            r = requests.get(url, headers=headers, params=params, timeout=30)
        except requests.RequestException:
            if attempt == retries:
                raise
            time.sleep(2 ** attempt)
            continue

        if r.status_code in (403, 429):
            if r.headers.get("X-RateLimit-Remaining") == "0":
                reset = int(r.headers.get("X-RateLimit-Reset", "0"))
                reset_at = datetime.fromtimestamp(reset, timezone.utc).strftime("%Y-%m-%d %H:%M:%S")
                raise GitHubRateLimit(f"rate limit esgotado; reseta as {reset_at} UTC")
            if r.headers.get("Retry-After") and attempt < retries:
                time.sleep(min(int(r.headers["Retry-After"]), 60))
                continue

        if r.status_code >= 500 and attempt < retries:
            time.sleep(2 ** attempt)
            continue

        r.raise_for_status()
        return r
    raise RuntimeError("falha apos retries")


def fetch_issues(repo, since_iso, token=None, max_pages=10, per_page=100):
    """Retorna (itens, truncado, chamadas_api, rate_remaining)."""
    headers = {
        "Accept": "application/vnd.github+json",
        "X-GitHub-Api-Version": "2022-11-28",
        "User-Agent": "incremental-pipeline-portfolio",
    }
    if token:
        headers["Authorization"] = f"Bearer {token}"

    url = f"{API}/repos/{repo}/issues"
    params = {"state": "all", "since": since_iso, "sort": "updated",
              "direction": "asc", "per_page": per_page}
    items, pages, remaining = [], 0, None

    while url and pages < max_pages:
        r = _get(url, headers, params)
        params = None  # o link "next" ja carrega a query completa
        items.extend(r.json())
        pages += 1
        remaining = r.headers.get("X-RateLimit-Remaining")
        url = r.links.get("next", {}).get("url")

    return items, url is not None, pages, remaining